# TODO 6.7.3：1.5B 与 3B 模型小规模质量对比

本 Notebook 保留 0.5B 的 80 条 pilot 结果作为控制组，在相同 80 条样本、相同 Prompt 和相同生成参数下依次运行 1.5B 与 3B。

请先把两个模型放到 `MODEL_ROOT` 下的目录；如果目录名不同，只需修改下一代码单元的 `MODEL_CANDIDATES`。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc
import json
import sys
from pathlib import Path

import torch

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_ROOT = Path('/content/drive/MyDrive/banking_llm_project/model_input_real')
BASELINE_CONFIG = PROJECT_DIR / 'configs/baseline.json'
QUALITY_CONFIG = PROJECT_DIR / 'configs/quality_repair.json'
RESULT_ROOT = PROJECT_DIR / 'out/quality_repair_6_7_3_model_scale'
sys.path.insert(0, str(PROJECT_DIR))

from src.baseline.service import BaselineService
from src.data.finqa_assets import load_assets
from src.evaluation.quality_repair import run_quality_repair_experiment

assert torch.cuda.is_available(), '请切换到 NVIDIA GPU。'

MODEL_CANDIDATES = {
    'qwen2.5-1.5b': [
        MODEL_ROOT / 'qwen2.5-1.5b-instruct',
        MODEL_ROOT / 'Qwen2.5-1.5B-Instruct',
    ],
    'qwen2.5-3b': [
        MODEL_ROOT / 'qwen2.5-3b-instruct',
        MODEL_ROOT / 'Qwen2.5-3B-Instruct',
    ],
}
MODEL_IDS = {
    'qwen2.5-1.5b': 'Qwen/Qwen2.5-1.5B-Instruct',
    'qwen2.5-3b': 'Qwen/Qwen2.5-3B-Instruct',
}

def resolve_model_dir(candidates):
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    expected = '\n'.join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f'找不到模型目录，请检查以下路径：\n{expected}')

def write_local_manifest(model_key, model_dir, output_dir):
    model_id = MODEL_IDS[model_key]
    manifest = {
        'schema_version': 1,
        'model_id': model_id,
        'model_path': str(model_dir),
        'version': model_id,
        'commit': 'colab-local',
        'weight_files': [],
        'config_file': 'config.json',
        'tokenizer_files': [],
        'expected_sha256': {},
        'runtime': {
            'framework': 'transformers',
            'model_class': 'causal_lm',
            'dtype': 'float16',
            'device': 'cuda:0',
            'trust_remote_code': False,
        },
    }
    manifest_path = output_dir / 'model_manifest.json'
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return manifest_path

assets = load_assets(DATASET_DIR)
quality_config = json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))
dataset_manifest = json.loads((DATASET_DIR / 'manifest.json').read_text(encoding='utf-8'))
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

all_reports = {}
for model_key, candidates in MODEL_CANDIDATES.items():
    model_dir = resolve_model_dir(candidates)
    model_result_dir = RESULT_ROOT / model_key
    model_result_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = write_local_manifest(model_key, model_dir, model_result_dir)
    print(f'\n===== {MODEL_IDS[model_key]} =====')
    print(f'模型目录: {model_dir}')
    print(f'结果目录: {model_result_dir}')
    service = BaselineService.from_local_model(
        BASELINE_CONFIG,
        manifest_path,
        model_result_dir / 'quality_service.jsonl',
    )
    report = run_quality_repair_experiment(
        service,
        assets.quality_dev,
        dataset_manifest,
        quality_config,
        model_result_dir,
        calibration=assets.calibration,
    )
    report['model'] = {
        'key': model_key,
        'model_id': MODEL_IDS[model_key],
        'model_path': str(model_dir),
    }
    (model_result_dir / 'comparison_report.json').write_text(
        json.dumps(report, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    all_reports[model_key] = report
    for variant in report['variants']:
        metrics = variant['metrics']
        print(
            f"{variant['variant']}: parse={metrics['parse_rate']:.2%}, "
            f"accuracy={metrics['numeric_accuracy']:.2%}, "
            f"correct={metrics['correct']}/{metrics['total']}"
        )
    del service
    gc.collect()
    torch.cuda.empty_cache()

print('\n===== 模型规模对比完成 =====')
for model_key, report in all_reports.items():
    print(model_key, '->', RESULT_ROOT / model_key / 'comparison_report.json')